In [129]:
import pickle
import hdbscan
import umap
import numpy as np
import matplotlib.pyplot as plt


In [130]:
embeddings_file = "dataset/embeddings.pkl" 
sentences_file = "dataset/sentences.txt"


In [131]:
sentences = []
with open(sentences_file, 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            sentences.append(line)
 

In [132]:
# load embeddings
with open(embeddings_file, 'rb') as f:
    embeddings = pickle.load(f)
 
# convert to numpy array just in case
embeddings = np.array(embeddings)

In [134]:
reducer = umap.UMAP(n_components=12, metric='cosine', random_state=42)
embeddings_2d = reducer.fit_transform(embeddings)


/home/prashanth/nlp/financial_assistant/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [135]:
clusterer = hdbscan.HDBSCAN(min_cluster_size=6, metric='euclidean')
labels = clusterer.fit_predict(embeddings_2d)

In [136]:
num_clusters = len(set(labels)) - (1 if -1 in labels else 0)
num_noise = list(labels).count(-1)

print(f"total points {len(sentences)}")
print(f"found {num_clusters} clusters")
print(f"noise points (unclustered): {num_noise}")

total points 3408
found 122 clusters
noise points (unclustered): 761


# parameter tunning

In [137]:
import optuna
import umap
import hdbscan
import numpy as np
from hdbscan.validity import validity_index

In [138]:
from sklearn.metrics import silhouette_score

n_samples, n_dims = embeddings.shape

def objective(trial):
    n_components     = trial.suggest_int("n_components",     2, min(n_dims, 50))
    min_cluster_size = trial.suggest_int("min_cluster_size", 2, max(5, n_samples // 20))

    reducer = umap.UMAP(n_components=n_components, metric='cosine', random_state=42)
    embeddings_2d = reducer.fit_transform(embeddings)

    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, metric='euclidean', gen_min_span_tree=True)
    labels = clusterer.fit_predict(embeddings_2d)

    unique_clusters = set(labels) - {-1}
    n_noise = int(np.sum(labels == -1))

    if len(unique_clusters) < 2:
        return -1.0, n_noise, -1.0

    dbcv_score = validity_index(embeddings_2d.astype(np.float64), labels)

    mask = labels != -1
    sil_score = silhouette_score(embeddings_2d[mask], labels[mask]) if mask.sum() > 1 else -1.0

    trial.set_user_attr("n_noise", n_noise)
    trial.set_user_attr("silhouette", sil_score)
    trial.set_user_attr("dbcv", dbcv_score)

    print(f"Trial {trial.number} | n_components={n_components} min_cluster_size={min_cluster_size} | noise={n_noise} sil={sil_score:.4f} dbcv={dbcv_score:.4f}")

    return dbcv_score  # primary metric to optimise

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print(study.best_params)
print(study.best_value)
print(study.best_trial.user_attrs)  # shows noise, silhouette, dbcv of best trial

[I 2026-04-12 23:42:48,022] A new study created in memory with name: no-name-52b0bfbd-361f-4895-84f7-0ac58b8100b9
/home/prashanth/nlp/financial_assistant/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
[I 2026-04-12 23:43:01,534] Trial 0 finished with value: 0.46124308464121067 and parameters: {'n_components': 40, 'min_cluster_size': 16}. Best is trial 0 with value: 0.46124308464121067.


Trial 0 | n_components=40 min_cluster_size=16 | noise=1053 sil=0.7032 dbcv=0.4612


/home/prashanth/nlp/financial_assistant/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
[I 2026-04-12 23:43:16,000] Trial 1 finished with value: 0.06027560784185696 and parameters: {'n_components': 46, 'min_cluster_size': 132}. Best is trial 0 with value: 0.46124308464121067.


Trial 1 | n_components=46 min_cluster_size=132 | noise=2724 sil=0.5119 dbcv=0.0603


/home/prashanth/nlp/financial_assistant/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
[I 2026-04-12 23:43:30,530] Trial 2 finished with value: 0.511294615403285 and parameters: {'n_components': 45, 'min_cluster_size': 8}. Best is trial 2 with value: 0.511294615403285.


...

Trial 46 | n_components=48 min_cluster_size=23 | noise=1095 sil=0.6882 dbcv=0.4391


/home/prashanth/nlp/financial_assistant/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
[W 2026-04-12 23:53:56,768] Trial 47 failed with parameters: {'n_components': 41, 'min_cluster_size': 169} because of the following error: The number of the values 3 did not match the number of the objectives 1.
[W 2026-04-12 23:53:56,768] Trial 47 failed with value (-1.0, 3408, -1.0).
/home/prashanth/nlp/financial_assistant/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
[I 2026-04-12 23:54:10,246] Trial 48 finished with value: 0.05964068240408857 and parameters: {'n_components': 41, 'min_cluster_size': 159}. Best is trial 26 with value: 0.5695960472472262.


Trial 48 | n_components=41 min_cluster_size=159 | noise=2835 sil=0.5508 dbcv=0.0596


/home/prashanth/nlp/financial_assistant/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
[I 2026-04-12 23:54:22,122] Trial 49 finished with value: 0.13226474959459378 and parameters: {'n_components': 2, 'min_cluster_size': 56}. Best is trial 26 with value: 0.5695960472472262.


Trial 49 | n_components=2 min_cluster_size=56 | noise=560 sil=0.4535 dbcv=0.1323
{'n_components': 43, 'min_cluster_size': 4}
0.5695960472472262
{'n_noise': 699, 'silhouette': 0.7101119160652161, 'dbcv': np.float64(0.5695960472472262)}


In [139]:
from hdbscan.validity import validity_index

n_samples, n_dims = embeddings.shape

def objective(trial):
    n_components     = trial.suggest_int("n_components",     2, min(n_dims, 50))
    min_cluster_size = trial.suggest_int("min_cluster_size", 2, max(5, n_samples // 20))

    reducer = umap.UMAP(n_components=n_components, metric='cosine', random_state=42)
    embeddings_2d = reducer.fit_transform(embeddings)

    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, metric='euclidean', gen_min_span_tree=True)
    labels = clusterer.fit_predict(embeddings_2d)

    n_noise = int(np.sum(labels == -1))
    trial.set_user_attr("n_noise", n_noise)

    if len(set(labels) - {-1}) < 2:
        return -1.0

    dbcv_score = validity_index(embeddings_2d.astype(np.float64), labels)
    print(f"Trial {trial.number} | n_components={n_components} min_cluster_size={min_cluster_size} | noise={n_noise} dbcv={dbcv_score:.4f}")

    return dbcv_score

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print(study.best_params)
print(study.best_value)
print("n_noise:", study.best_trial.user_attrs["n_noise"])

[I 2026-04-13 00:15:46,375] A new study created in memory with name: no-name-23c1bc0c-3c5f-4bed-8d69-6edc9b7a028b
/home/prashanth/nlp/financial_assistant/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
[I 2026-04-13 00:15:59,605] Trial 0 finished with value: 0.23030940163393965 and parameters: {'n_components': 12, 'min_cluster_size': 107}. Best is trial 0 with value: 0.23030940163393965.


Trial 0 | n_components=12 min_cluster_size=107 | noise=210 dbcv=0.2303


/home/prashanth/nlp/financial_assistant/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
[I 2026-04-13 00:16:13,662] Trial 1 finished with value: 0.11612906488776954 and parameters: {'n_components': 22, 'min_cluster_size': 89}. Best is trial 0 with value: 0.23030940163393965.


Trial 1 | n_components=22 min_cluster_size=89 | noise=203 dbcv=0.1161


/home/prashanth/nlp/financial_assistant/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
[I 2026-04-13 00:16:25,526] Trial 2 finished with value: 0.5310040189970274 and parameters: {'n_components': 8, 'min_cluster_size': 6}. Best is trial 2 with value: 0.5310040189970274.


...

Trial 47 | n_components=42 min_cluster_size=46 | noise=606 dbcv=0.3824


/home/prashanth/nlp/financial_assistant/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
[I 2026-04-13 00:27:19,868] Trial 48 finished with value: 0.3766318218625862 and parameters: {'n_components': 9, 'min_cluster_size': 21}. Best is trial 33 with value: 0.5646246856898669.


Trial 48 | n_components=9 min_cluster_size=21 | noise=1081 dbcv=0.3766


/home/prashanth/nlp/financial_assistant/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
[I 2026-04-13 00:27:34,650] Trial 49 finished with value: 0.4996122023353986 and parameters: {'n_components': 47, 'min_cluster_size': 11}. Best is trial 33 with value: 0.5646246856898669.


Trial 49 | n_components=47 min_cluster_size=11 | noise=869 dbcv=0.4996
{'n_components': 43, 'min_cluster_size': 3}
0.5646246856898669
n_noise: 641
